# **Imports**


In [15]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, hamming_loss

# Laad de datasets
X_train = pd.read_csv('Datasets/cleaned_X_train2.csv')
X_test = pd.read_csv('Datasets/cleaned_X_test2.csv')
y_train = pd.read_csv('Datasets/y_train.csv')
y_test = pd.read_csv('Datasets/y_test.csv')

# Imputeer missende waarden
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Maak het model
logreg = LogisticRegression(max_iter=1000, multi_class='ovr', solver='liblinear', class_weight='balanced')

# Gebruik MultiOutputClassifier om meerdere labels te voorspellen
multi_target_model = MultiOutputClassifier(logreg)

# Hyperparameter tuning met GridSearchCV
param_grid = {
    'estimator__C': [0.1, 1.0, 10.0],  # Regel de regularisatie
    'estimator__max_iter': [500, 1000],  # Regel het aantal iteraties
    'estimator__solver': ['liblinear', 'saga']  # Verschillende solvers proberen
}

grid_search = GridSearchCV(multi_target_model, param_grid, cv=5, n_jobs=-1, verbose=2)

# Train het model met GridSearchCV
grid_search.fit(X_train, y_train)

# Beste parameters van GridSearchCV
print(f"Beste parameters: {grid_search.best_params_}")

# Gebruik het beste model
best_model = grid_search.best_estimator_

# Voorspellingen maken
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Evaluatie: Train en Test accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

# Evaluatie: Hamming loss
train_hamming_loss = hamming_loss(y_train, y_pred_train)
test_hamming_loss = hamming_loss(y_test, y_pred_test)

# Print resultaten
print(f"Train Accuracy: {train_accuracy}")
print(f"Test Accuracy: {test_accuracy}")
print(f"Train Hamming Loss: {train_hamming_loss}")
print(f"Test Hamming Loss: {test_hamming_loss}")

# Cross-validation voor robuustere resultaten
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Cross-validated accuracy: {cv_scores.mean()}")




Fitting 5 folds for each of 12 candidates, totalling 60 fits


/home/iyoas/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: Con

Beste parameters: {'estimator__C': 10.0, 'estimator__max_iter': 500, 'estimator__solver': 'liblinear'}
Train Accuracy: 0.5346534653465347
Test Accuracy: 0.019801980198019802
Train Hamming Loss: 0.10173267326732673
Test Hamming Loss: 0.2801980198019802
Cross-validated accuracy: 0.09645061728395062
